In [31]:
from DecisionTree import DecisionTree
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [49]:
#define helper functions
#function to evaluate model's accuracy
def evaluate_acc(y_true, y_pred):
    total_samples = len(y_true)
    correct_predictions = np.sum(y_true == y_pred)
    return (correct_predictions / total_samples) 

#function to create train test split
def create_train_test_split(x, y, test_ratio):
    n_samples = x.shape[0]

    # shuffle indices
    indices = np.random.permutation(n_samples)

    # compute split point
    test_count = int(n_samples * test_ratio)

    test_idx = indices[:test_count] #get indices for test set
    train_idx = indices[test_count:] #get indices for train set

    #create train and test sets
    x_train = x.iloc[train_idx]
    x_test = x.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]


    return x_train, x_test, y_train, y_test

def get_pca_projection(X, n_components=2):
    #standardize
    X_std = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

    #covariance matrix
    cov_mat = np.cov(X_std.T)

    #eigenvalues and eigenvectors
    eig_vals, eig_vecs = np.linalg.eig(cov_mat)

    #get top eigenvectors
    idx = np.argsort(eig_vals)[::-1]
    top_vectors = eig_vecs[:, idx[:n_components]]

    # Return the projected data and the vectors 
    return X_std.dot(top_vectors), top_vectors

def plot_decision_boundary(model, X, y, title):
    #get 2Dfeatures
    X_arr = np.array(X)
    X_2d, _ = get_pca_projection(X_arr, n_components=2)
    y_arr = np.array(y)

    #create mesh grid
    h = 1.0  # Keep this large for faster plotting
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), 
                         np.arange(y_min, y_max, h))

    # Train a new kNN model on the 2D data for fast prediction
    model_2d = DecisionTree()
    model_2d.fit(X_2d, y)
    
    #Predictions on the grid points
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = np.array(model_2d.predict(grid_points))
    Z = Z.reshape(xx.shape)

    #Plotting
    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, Z,levels=[-0.5, 0.5, 1.5], alpha=0.3, cmap='coolwarm')
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, edgecolor='k', s=20, cmap='coolwarm')
    plt.xlabel("PC1")   
    plt.ylabel("PC2")
    plt.title(title)

    # 6. Save the file
    folder = "DT_plots"
    if not os.path.exists(folder):
        os.makedirs(folder)
    
    filename = title.replace(" ", "_").replace(":", "").replace(",", "")
    save_path = f"{folder}/{filename}.png"
    
    plt.savefig(save_path)
    plt.close() # Free up memory
    print(f"Plot saved to: {save_path}")

In [33]:
#load datasets
breast_cancer_df = pd.read_csv(r"data_cleaned\breast_cancer.csv")
hepatitis_df = pd.read_csv(r"data_cleaned\hepatitis.csv")
hepatitis_imputated_df = pd.read_csv(r"data_cleaned\hepatitis_imputed.csv")

#create training and testing sets
#breast cancer dataset
x_bc = breast_cancer_df.drop(columns="diagnosis")
y_bc = breast_cancer_df["diagnosis"]
x_train_bc, x_test_bc, y_train_bc, y_test_bc = create_train_test_split(x_bc, y_bc, 0.2)

#hepatitis dataset
x_hp = hepatitis_df.drop(columns="class")
y_hp = hepatitis_df["class"]
x_train_hp, x_test_hp, y_train_hp, y_test_hp = create_train_test_split(x_hp, y_hp, 0.2)

#hepatitis dataset with imputation
x_hp_imp = hepatitis_imputated_df.drop(columns="class")
y_hp_imp = hepatitis_imputated_df["class"]
x_train_hp_imp, x_test_hp_imp, y_train_hp_imp, y_test_hp_imp = create_train_test_split(x_hp_imp, y_hp_imp, 0.2)

In [ ]:
#Experiments: Check how different different min_sample_leaf values affect the performance

datasets = {
    "bc": (x_train_bc, y_train_bc, x_test_bc, y_test_bc),
    "hp": (x_train_hp, y_train_hp, x_test_hp, y_test_hp),
    "hpimp": (x_train_hp_imp, y_train_hp_imp, x_test_hp_imp, y_test_hp_imp)
}

hyperparams = {
    "max_depth": [1,2,3,4,5,6,7,8,9,10],
    "criterion": ["gini", "entropy"],
    "min_samples_leaf": [1,2,5,10],
    "min_samples_split": [1,2,5,10]
}

# helper function to save predictions
def save_predictions(model, X, y, dataset_name, kind, param_name, param_value):
    preds = model.predict(X)
    df = pd.DataFrame({"y_test": y, "y_pred": preds})
    df.to_csv(f"predictions_dt/{dataset_name}_predictions_{param_name}_{param_value}_{kind}.csv", index=False)

# loop over hyperparameters
for param_name, values in hyperparams.items():
    for val in values:
        for dataset_name, (X_train, y_train, X_test, y_test) in datasets.items():
            # set model arguments dynamically
            kwargs = {param_name: val}
            # keep default values for others
            if "max_depth" not in kwargs:
                kwargs["max_depth"] = 3
            if "criterion" not in kwargs:
                kwargs["criterion"] = "gini"
            if "min_samples_leaf" not in kwargs:
                kwargs["min_samples_leaf"] = 1
            if "min_samples_split" not in kwargs:
                kwargs["min_samples_split"] = 2
            
            # train
            model = DecisionTree(**kwargs)
            model.fit(X_train, y_train)
            
            # save train and test predictions
            save_predictions(model, X_train, y_train, dataset_name, "train", param_name, val)
            save_predictions(model, X_test, y_test, dataset_name, "test", param_name, val)



[[-0.21425955 -0.2405155 ]
 [-0.10436033 -0.0647405 ]
 [-0.22356568 -0.22057498]
 [-0.21655418 -0.23516639]
 [-0.14574587  0.18255985]
 [-0.23799619  0.14964555]
 [-0.25521008  0.0677919 ]
 [-0.25965665 -0.02660232]
 [-0.14311147  0.18256062]
 [-0.07010925  0.36625905]
 [-0.20760649 -0.07860938]
 [-0.02564151  0.1036712 ]
 [-0.21697807 -0.06193124]
 [-0.21864501 -0.14320976]
 [-0.018703    0.22500606]
 [-0.16971828  0.23159947]
 [-0.15484241  0.19880725]
 [-0.18014273  0.1397123 ]
 [-0.04263484  0.20157377]
 [-0.0989889   0.28341175]
 [-0.22435997 -0.22904263]
 [-0.10574388 -0.06012854]
 [-0.23385879 -0.20721714]
 [-0.22061472 -0.22669493]
 [-0.13246719  0.15716882]
 [-0.20916751  0.12701454]
 [-0.22806143  0.08265747]
 [-0.24951052 -0.01423087]
 [-0.1277434   0.11868483]
 [-0.13373376  0.2600157 ]]
Plot saved to: DT_plots/bc_predictions_max_depth_1.png
[[-0.11798483  0.21426905]
 [-0.02518448  0.1841221 ]
 [-0.0135249  -0.22569443]
 [ 0.2155649  -0.14024396]
 [-0.25614857 -0.4081169 ]

KeyboardInterrupt: 

In [50]:
#calculate final metrics using the best Decision Tree model
for dataset_name, (X_train, y_train, X_test, y_test) in datasets.items():
    model = DecisionTree(max_depth=4, criterion='entropy', min_samples_leaf=1, min_samples_split=2)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    TP = np.sum((y_test == 1) & (y_pred == 1))
    TN = np.sum((y_test == 0) & (y_pred == 0))
    FP = np.sum((y_test == 0) & (y_pred == 1))
    FN = np.sum((y_test == 1) & (y_pred == 0))

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

    plot_decision_boundary(model, X_train, y_train, f"{dataset_name}_predictions_{param_name}_{val}_train")

    print(f"\n{dataset_name} results")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Sensitivity:", sensitivity)
    print("Specificity:", specificity)
    print("F1:", f1)


Plot saved to: DT_plots/bc_predictions_max_depth_2_train.png

bc results
Accuracy: 0.9292035398230089
Precision: 0.9743589743589743
Sensitivity: 0.8444444444444444
Specificity: 0.9852941176470589
F1: 0.9047619047619048
Plot saved to: DT_plots/hp_predictions_max_depth_2_train.png

hp results
Accuracy: 0.75
Precision: 0.0
Sensitivity: 0.0
Specificity: 0.8
F1: 0
Plot saved to: DT_plots/hpimp_predictions_max_depth_2_train.png

hpimp results
Accuracy: 0.8064516129032258
Precision: 0.5714285714285714
Sensitivity: 0.5714285714285714
Specificity: 0.875
F1: 0.5714285714285714
